# Effect of One Threshold-Link Agent with a 10% Activation Share

This notebook analyzes the paired threshold-link run produced by the current configuration. For every network size and parameter row, it compares an all-best-response baseline with a treatment containing one threshold-link agent. In the treatment, that agent receives 10% of scheduler selections; this bundles the policy change with unequal activation frequency. Time is measured in population-average activations per agent, not equal opportunities for every individual.

In [ ]:
import json
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import zarr
from scipy.stats import spearmanr
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.inspection import permutation_importance
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.model_selection import train_test_split

ROOT = Path.cwd().resolve()
if not (ROOT / "tasks").is_dir():
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from network_creation.policy_result_metadata import (
    configured_n_actors,
    configured_output_step_labels,
)
from network_creation.time_normalization import (
    common_normalized_time_grid,
    interpolate_time_axis,
    normalized_source_times,
)
from tasks.policies import load_policy_results

RUN = ROOT / "artifacts/simulation_runs/run-20260727T162050.zarr"
CONFIG = ROOT / "configs/best_response_threshold_link.one_threshold.run_config.json"

COMPOSITION_AXIS = "threshold_link_count"
BASELINE_COUNT = 0
TREATMENT_COUNT = 1
PRACTICAL_EFFECT = 0.05
SIZE_COLORS = {10: "#3d405b", 50: "#e07a5f", 100: "#2a9d8f"}

plt.style.use("seaborn-v0_8-whitegrid")
pd.set_option("display.max_columns", 20)

## 1. Experimental Design

The continuous behavioral parameters use a shared scrambled Sobol design, and visibility is sampled from the six supported integer values. Each treatment row has the same parameter row and simulator seed as its baseline counterpart, although unequal activation weights mean that the two runs do not share an identical sequence of activated agents. The artifact's embedded configuration is checked against the current checked-in config before analysis.

In [ ]:
run_config = json.loads((RUN / "run_config.json").read_text())
checked_config = json.loads(CONFIG.read_text())
contract_sections = ("sampling", "simulator", "game", "population", "observers")
if any(run_config[name] != checked_config[name] for name in contract_sections):
    raise ValueError("The selected artifact was not produced by the current config")

metadata = load_policy_results(RUN, stats=())
if metadata.attrs["has_failures"]:
    raise ValueError("The source run contains failed rows")

n_agents = metadata.coords["n_agents"].to_numpy().astype(int)
composition_values = metadata.coords[COMPOSITION_AXIS].to_numpy().astype(int)
if not np.array_equal(composition_values, [BASELINE_COUNT, TREATMENT_COUNT]):
    raise ValueError(f"Expected threshold-link counts [0, 1], got {composition_values}")
composition_index = {value: index for index, value in enumerate(composition_values)}
BASELINE = composition_index[BASELINE_COUNT]
TREATMENT = composition_index[TREATMENT_COUNT]
n_parameter_rows = metadata.sizes["parameter_row"]
n_compositions = metadata.sizes[COMPOSITION_AXIS]

sampled_names = [
    "visibility_distance",
    "threshold_link.link_threshold",
    "best_response.outside_optimism",
    "best_response.exploration",
]
paired_parameters = metadata["theta"].sel(theta_parameter=sampled_names).to_numpy()
if not np.array_equal(
    paired_parameters[:, BASELINE], paired_parameters[:, TREATMENT]
):
    raise ValueError("Treatment and baseline parameter rows do not match")

root = zarr.open_group(RUN, mode="r")
input_coordinates = np.asarray(root["metadata/input_coordinates"], dtype=int)
result_input_index = np.asarray(root["results/input_index"], dtype=int)
output_step_labels = configured_output_step_labels(run_config)
n_actors = configured_n_actors(run_config)

def extract_normalized_output(
    output_name: str, normalized_times: np.ndarray
) -> np.ndarray:
    """Interpolate one output in Zarr-sized blocks onto normalized time."""
    output = root[f"results/outputs/{output_name}"]
    raw_steps = output_step_labels[output_name]
    if output.shape[1] != len(raw_steps):
        raise ValueError(f"Configured steps do not match {output_name!r}")
    values = np.full(
        (
            len(n_agents),
            n_compositions,
            n_parameter_rows,
            len(normalized_times),
            *output.shape[2:],
        ),
        np.nan,
        dtype=np.float32,
    )

    chunk_rows = output.chunks[0]
    for start in range(0, output.shape[0], chunk_rows):
        stop = min(start + chunk_rows, output.shape[0])
        coordinates = input_coordinates[result_input_index[start:stop]]
        block = np.asarray(output[start:stop], dtype=np.float32)
        for size_index, size in enumerate(n_agents):
            rows = np.flatnonzero(coordinates[:, 0] == size_index)
            if not len(rows):
                continue
            selected = interpolate_time_axis(
                block[rows],
                source_times=normalized_source_times(
                    raw_steps, n_agents=size, n_actors=n_actors
                ),
                target_times=normalized_times,
                step_axis=1,
            )
            selected_coordinates = coordinates[rows]
            values[
                selected_coordinates[:, 0],
                selected_coordinates[:, 1],
                selected_coordinates[:, 2],
            ] = selected

    return values

welfare_output = "centered_mean_welfare.mean"
normalized_grid = common_normalized_time_grid(
    output_step_labels[welfare_output],
    n_agents_values=n_agents,
    n_actors=n_actors,
)
normalized_time = normalized_grid.step_labels
threshold_policy = next(
    policy
    for policy in run_config["population"]["policies"]
    if policy["id"] == "threshold_link"
)
threshold_activation_share = threshold_policy["agent_parameters"]["activation_weight"]["share"]["value"]

design_summary = pd.DataFrame(
    {
        "Design item": [
            "Network sizes",
            "Population compositions",
            "Shared parameter rows",
            "Episodes per configuration",
            "Raw simulation steps",
            "Common normalized horizon",
            "Threshold-agent activation share",
            "Stored trajectories",
        ],
        "Value": [
            ", ".join(map(str, n_agents)),
            "all best-response; one threshold-link agent",
            f"{n_parameter_rows:,}",
            run_config["simulator"]["n_episodes"],
            run_config["simulator"]["max_steps"],
            f"{normalized_time[-1]:g} population-average activations per agent",
            f"{threshold_activation_share:.0%} of treatment selections",
            "welfare, indegree, centralization, policy payoff",
        ],
    }
)

exposure_summary = pd.DataFrame(
    {
        "n_agents": n_agents,
        "Baseline BR agent": normalized_time[-1],
        "Treatment threshold agent": (
            normalized_time[-1] * n_agents * threshold_activation_share
        ),
        "Treatment BR peer": (
            normalized_time[-1]
            * n_agents
            * (1 - threshold_activation_share)
            / (n_agents - 1)
        ),
    }
).set_index("n_agents")

display(design_summary)
display(exposure_summary.style.format("{:.1f}"))

## 2. Paired Welfare Effects

For size $n$, parameter row $j$, and normalized time $t$, the paired treatment contrast is

$$
\Delta W_{nj}(t)=W_{n,\text{threshold treatment},j}(t)-W_{n,\text{all BR},j}(t).
$$

The 5–95% ranges below describe heterogeneity across the sampled parameter space; they are not confidence intervals. Because policy assignment and activation share change together, $\Delta W$ is not a policy-only replacement effect.

In [ ]:
welfare = extract_normalized_output(
    welfare_output, normalized_time
)
welfare_delta = welfare[:, TREATMENT] - welfare[:, BASELINE]
baseline_welfare = welfare[:, BASELINE, :, -1].copy()
threshold_welfare = welfare[:, TREATMENT, :, -1].copy()
del welfare

def parameter_vector(name: str) -> np.ndarray:
    return (
        metadata["theta"]
        .sel(theta_parameter=name)
        .isel(n_agents=0, **{COMPOSITION_AXIS: BASELINE})
        .to_numpy()
        .copy()
    )

theta = pd.DataFrame(
    {
        "parameter_row": np.arange(n_parameter_rows),
        "visibility": parameter_vector("visibility_distance").astype(int),
        "link_threshold": parameter_vector("threshold_link.link_threshold"),
        "outside_optimism": parameter_vector("best_response.outside_optimism"),
        "exploration": parameter_vector("best_response.exploration"),
    }
)

effect_frames = []
for size_index, size in enumerate(n_agents):
    frame = theta.copy()
    frame["n_agents"] = size
    frame["d_welfare"] = welfare_delta[size_index, :, -1]
    effect_frames.append(frame)
effects = pd.concat(effect_frames, ignore_index=True)

In [ ]:
time_average_effect = np.trapezoid(
    welfare_delta, normalized_time, axis=-1
) / normalized_time[-1]

summary_rows = []
for size_index, size in enumerate(n_agents):
    delta = welfare_delta[size_index, :, -1]
    q05, median, q95 = np.quantile(delta, [0.05, 0.50, 0.95])
    visibility_stratified = np.mean(
        [delta[theta.visibility == value].mean() for value in range(1, 7)]
    )
    summary_rows.append(
        {
            "n_agents": size,
            "All-BR welfare": baseline_welfare[size_index].mean(),
            "Treatment welfare": threshold_welfare[size_index].mean(),
            "Mean effect": delta.mean(),
            "Visibility-stratified mean": visibility_stratified,
            "Median effect": median,
            "5%-95% parameter range": f"[{q05:.4f}, {q95:.4f}]",
            "P(effect > 0)": np.mean(delta > 0),
            f"P(effect > {PRACTICAL_EFFECT:g})": np.mean(
                delta > PRACTICAL_EFFECT
            ),
            f"P(effect < -{PRACTICAL_EFFECT:g})": np.mean(
                delta < -PRACTICAL_EFFECT
            ),
            "Time-average effect": time_average_effect[size_index].mean(),
        }
    )

welfare_summary = pd.DataFrame(summary_rows).set_index("n_agents")
welfare_summary.style.format(
    {
        "All-BR welfare": "{:.4f}",
        "Treatment welfare": "{:.4f}",
        "Mean effect": "{:.4f}",
        "Visibility-stratified mean": "{:.4f}",
        "Median effect": "{:.4f}",
        "P(effect > 0)": "{:.1%}",
        f"P(effect > {PRACTICAL_EFFECT:g})": "{:.1%}",
        f"P(effect < -{PRACTICAL_EFFECT:g})": "{:.1%}",
        "Time-average effect": "{:.4f}",
    }
)

## 3. Effect Dynamics

The endpoint does not describe when an effect appears or whether it reverses. The solid line is the mean paired effect, the dashed line is the median, and the band is the 5–95% parameter-row range.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4), sharey=True, constrained_layout=True)
for size_index, (size, ax) in enumerate(zip(n_agents, axes, strict=True)):
    delta = welfare_delta[size_index]
    q05, median, q95 = np.quantile(delta, [0.05, 0.50, 0.95], axis=0)
    mean = delta.mean(axis=0)
    color = SIZE_COLORS[size]
    ax.fill_between(normalized_time, q05, q95, color=color, alpha=0.18)
    ax.plot(normalized_time, mean, color=color, lw=2.2, label="mean")
    ax.plot(normalized_time, median, color=color, lw=1.5, ls="--", label="median")
    ax.axhline(0, color="black", lw=0.8)
    ax.set_title(f"n = {size}")
    ax.set_xlabel("population-average activations per agent")
axes[0].set_ylabel("paired welfare effect")
axes[0].legend(frameon=False)
fig.suptitle("Threshold-link treatment effect over time")
plt.show()

In [ ]:
dynamic_rows = []
for size_index, size in enumerate(n_agents):
    delta = welfare_delta[size_index]
    mean_curve = delta.mean(axis=0)
    row_average = time_average_effect[size_index]
    peak_index = int(np.argmax(mean_curve))
    trough_index = int(np.argmin(mean_curve))
    dynamic_rows.append(
        {
            "n_agents": size,
            "Time-average effect": row_average.mean(),
            "Median time-average effect": np.median(row_average),
            "P(time-average effect > 0)": np.mean(row_average > 0),
            "Peak time": normalized_time[peak_index],
            "Peak effect": mean_curve[peak_index],
            "Trough time": normalized_time[trough_index],
            "Trough effect": mean_curve[trough_index],
            "Endpoint effect": mean_curve[-1],
        }
    )

dynamic_summary = pd.DataFrame(dynamic_rows).set_index("n_agents")
dynamic_summary.style.format(
    {
        "Time-average effect": "{:.4f}",
        "Median time-average effect": "{:.4f}",
        "P(time-average effect > 0)": "{:.1%}",
        "Peak time": "{:.1f}",
        "Peak effect": "{:.4f}",
        "Trough time": "{:.1f}",
        "Trough effect": "{:.4f}",
        "Endpoint effect": "{:.4f}",
    }
)

## 4. Parameter-Space Heterogeneity

`link_threshold` is a proportion of locally possible outgoing links, not an action probability (`p_act` is fixed at one). Its interpretation therefore depends directly on visibility. The threshold agent's activation share is fixed at 10%, so it is not part of the sampled response surface. The following figures use empirical conditional means rather than a linear threshold slope.

In [ ]:
profile_specs = {
    "link_threshold": (np.linspace(0, 1, 11), "link threshold"),
    "visibility": (np.arange(0.5, 7.5, 1), "visibility distance"),
    "outside_optimism": (np.linspace(0, 1, 11), "outside optimism"),
    "exploration": (np.linspace(0.0001, 0.1, 11), "exploration"),
}

fig, axes = plt.subplots(2, 2, figsize=(12, 8), constrained_layout=True)
for (feature, (edges, label)), ax in zip(profile_specs.items(), axes.flat, strict=True):
    for size in n_agents:
        selected = effects.loc[effects.n_agents == size].copy()
        selected["bin"] = pd.cut(selected[feature], edges, include_lowest=True)
        profile = selected.groupby("bin", observed=True).agg(
            x=(feature, "mean"),
            mean_effect=("d_welfare", "mean"),
        )
        ax.plot(
            profile.x,
            profile.mean_effect,
            marker="o",
            ms=4,
            color=SIZE_COLORS[size],
            label=f"n = {size}",
        )
    ax.axhline(0, color="black", lw=0.8)
    ax.set_xlabel(label)
    ax.set_ylabel("mean paired welfare effect")
axes[0, 0].legend(frameon=False)
fig.suptitle("Marginal treatment-effect profiles at the common endpoint")
plt.show()

In [ ]:
threshold_edges = np.linspace(0, 1, 11)
visibility_edges = np.arange(0.5, 7.5, 1)
threshold_visibility_grids = []
for size in n_agents:
    selected = effects.loc[effects.n_agents == size].copy()
    selected["threshold_bin"] = pd.cut(
        selected.link_threshold, threshold_edges, labels=False, include_lowest=True
    )
    selected["visibility_bin"] = pd.cut(
        selected.visibility, visibility_edges, labels=False, include_lowest=True
    )
    grid = selected.pivot_table(
        index="visibility_bin",
        columns="threshold_bin",
        values="d_welfare",
        aggfunc="mean",
        observed=True,
    ).reindex(index=range(6), columns=range(10))
    threshold_visibility_grids.append(grid.to_numpy())

color_limit = max(np.nanmax(np.abs(grid)) for grid in threshold_visibility_grids)
fig, axes = plt.subplots(1, 3, figsize=(14, 4), constrained_layout=True)
for size, grid, ax in zip(n_agents, threshold_visibility_grids, axes, strict=True):
    image = ax.imshow(
        grid, origin="lower", aspect="auto", cmap="RdBu_r",
        vmin=-color_limit, vmax=color_limit,
    )
    ax.set_title(f"n = {size}")
    ax.set_xlabel("link-threshold decile")
    ax.set_xticks(range(10), range(1, 11))
    ax.set_yticks(range(6), range(1, 7))
axes[0].set_ylabel("visibility distance")
fig.colorbar(image, ax=axes, label="mean paired welfare effect")
fig.suptitle("Threshold × visibility interaction")
plt.show()

In [ ]:
optimism_edges = np.linspace(0, 1, 11)
threshold_optimism_grids = []
for size in n_agents:
    selected = effects.loc[effects.n_agents == size].copy()
    selected["threshold_bin"] = pd.cut(
        selected.link_threshold, threshold_edges, labels=False, include_lowest=True
    )
    selected["optimism_bin"] = pd.cut(
        selected.outside_optimism, optimism_edges, labels=False, include_lowest=True
    )
    grid = selected.pivot_table(
        index="optimism_bin",
        columns="threshold_bin",
        values="d_welfare",
        aggfunc="mean",
        observed=True,
    ).reindex(index=range(10), columns=range(10))
    threshold_optimism_grids.append(grid.to_numpy())

color_limit = max(np.nanmax(np.abs(grid)) for grid in threshold_optimism_grids)
fig, axes = plt.subplots(1, 3, figsize=(14, 4), constrained_layout=True)
for size, grid, ax in zip(n_agents, threshold_optimism_grids, axes, strict=True):
    image = ax.imshow(
        grid, origin="lower", aspect="auto", cmap="RdBu_r",
        vmin=-color_limit, vmax=color_limit,
    )
    ax.set_title(f"n = {size}")
    ax.set_xlabel("link-threshold decile")
    ax.set_xticks(range(10), range(1, 11))
    ax.set_yticks(range(10), range(1, 11))
axes[0].set_ylabel("outside-optimism decile")
fig.colorbar(image, ax=axes, label="mean paired welfare effect")
fig.suptitle("Threshold × outside-optimism interaction")
plt.show()

## 5. Nonlinear Response-Surface Diagnostic

A held-out surrogate checks how much of the effect surface can be recovered from the four sampled inputs. The same parameter rows are held out for every network size. The model is descriptive: its purpose is to expose nonlinearities and interactions, not to generate hypothesis-test p-values.

In [ ]:
feature_names = [
    "visibility", "link_threshold", "outside_optimism", "exploration"
]
X = theta[feature_names].to_numpy(dtype=float)
train_rows, test_rows = train_test_split(
    np.arange(n_parameter_rows), test_size=0.2, random_state=2026
)

performance_rows = []
importance_rows = []
for size_index, size in enumerate(n_agents):
    y = welfare_delta[size_index, :, -1].astype(float)

    linear = LinearRegression().fit(
        X[train_rows, 1:2], y[train_rows]
    )
    linear_prediction = linear.predict(X[test_rows, 1:2])
    performance_rows.append(
        {
            "n_agents": size,
            "model": "threshold-only linear",
            "Held-out R2": r2_score(y[test_rows], linear_prediction),
            "Held-out MAE": mean_absolute_error(y[test_rows], linear_prediction),
        }
    )

    nonlinear = HistGradientBoostingRegressor(
        max_iter=300, l2_regularization=1e-3, random_state=2026
    ).fit(X[train_rows], y[train_rows])
    nonlinear_prediction = nonlinear.predict(X[test_rows])
    performance_rows.append(
        {
            "n_agents": size,
            "model": "nonlinear interactions",
            "Held-out R2": r2_score(y[test_rows], nonlinear_prediction),
            "Held-out MAE": mean_absolute_error(y[test_rows], nonlinear_prediction),
        }
    )

    importance = permutation_importance(
        nonlinear,
        X[test_rows],
        y[test_rows],
        scoring="r2",
        n_repeats=10,
        random_state=2026,
    )
    for feature, mean, std in zip(
        feature_names, importance.importances_mean, importance.importances_std, strict=True
    ):
        importance_rows.append(
            {
                "n_agents": size,
                "feature": feature,
                "permutation_R2_loss": mean,
                "permutation_sd": std,
            }
        )

model_performance = pd.DataFrame(performance_rows)
model_performance.style.format(
    {"Held-out R2": "{:.3f}", "Held-out MAE": "{:.4f}"}
)

In [ ]:
importance_table = pd.DataFrame(importance_rows)
importance_pivot = importance_table.pivot(
    index="feature", columns="n_agents", values="permutation_R2_loss"
).loc[feature_names]
importance_pivot.plot.bar(
    figsize=(10, 4),
    color=[SIZE_COLORS[size] for size in n_agents],
    width=0.8,
)
plt.ylabel("held-out R² loss after permutation")
plt.xlabel("sampled input")
plt.title("Nonlinear surrogate permutation importance")
plt.xticks(rotation=20, ha="right")
plt.legend(title="n_agents", frameon=False)
plt.tight_layout()
plt.show()

## 6. Network Mechanisms

Mean indegree measures link abundance. Central point of dominance measures concentration around a high-betweenness node (zero for the empty network and one for a star). Associations with welfare are mechanism-consistent descriptions, not causal mediation estimates. They may reflect both the threshold decision rule and its unequal activation frequency.

In [ ]:
mechanism_rows = []
mechanism_curves = {}

for output_name, label in [
    ("indegree.mean", "mean indegree"),
    ("central_point_of_dominance.mean", "central point of dominance"),
]:
    values = extract_normalized_output(output_name, normalized_time)
    delta = values[:, TREATMENT] - values[:, BASELINE]
    mechanism_curves[label] = delta.mean(axis=1)

    for size_index, size in enumerate(n_agents):
        terminal_delta = delta[size_index, :, -1]
        q05, median, q95 = np.quantile(terminal_delta, [0.05, 0.50, 0.95])
        mechanism_rows.append(
            {
                "n_agents": size,
                "outcome": label,
                "All-BR mean": values[size_index, BASELINE, :, -1].mean(),
                "Treatment mean": values[size_index, TREATMENT, :, -1].mean(),
                "Mean change": terminal_delta.mean(),
                "Median change": median,
                "5%-95% parameter range": f"[{q05:.4f}, {q95:.4f}]",
                "Spearman rho with welfare effect": spearmanr(
                    terminal_delta, welfare_delta[size_index, :, -1]
                ).statistic,
            }
        )
    del values, delta

mechanism_summary = pd.DataFrame(mechanism_rows).set_index(
    ["n_agents", "outcome"]
)
mechanism_summary.style.format(
    {
        "All-BR mean": "{:.4f}",
        "Treatment mean": "{:.4f}",
        "Mean change": "{:.4f}",
        "Median change": "{:.4f}",
        "Spearman rho with welfare effect": "{:.3f}",
    }
)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4), constrained_layout=True)
for (label, curves), ax in zip(mechanism_curves.items(), axes, strict=True):
    for size_index, size in enumerate(n_agents):
        ax.plot(
            normalized_time,
            curves[size_index],
            color=SIZE_COLORS[size],
            lw=2,
            label=f"n = {size}",
        )
    ax.axhline(0, color="black", lw=0.8)
    ax.set_title(label)
    ax.set_xlabel("population-average activations per agent")
    ax.set_ylabel("mean paired change")
axes[0].legend(frameon=False)
fig.suptitle("How the treatment changes network structure")
plt.show()

## 7. Distribution of Benefits and Costs

Policy-specific payoff shows how benefits and link costs are distributed between the threshold agent and best-response peers. It uses a different normalization from aggregate welfare, so the two scales are reported separately. The comparison also reflects the treatment's unequal activation exposure.

In [ ]:
policy_welfare = extract_normalized_output(
    "policy_welfare.mean", np.asarray([normalized_time[-1]])
)
policy_class_index = {
    policy["id"]: index
    for index, policy in enumerate(run_config["population"]["policies"])
}
threshold_agent_payoff = policy_welfare[
    :, TREATMENT, :, 0, policy_class_index["threshold_link"]
]
treated_BR_payoff = policy_welfare[
    :, TREATMENT, :, 0, policy_class_index["best_response"]
]
baseline_BR_payoff = policy_welfare[
    :, BASELINE, :, 0, policy_class_index["best_response"]
]
del policy_welfare

policy_rows = []
for size_index, size in enumerate(n_agents):
    threshold_payoff = threshold_agent_payoff[size_index]
    treated_peer_payoff = treated_BR_payoff[size_index]
    baseline_peer_payoff = baseline_BR_payoff[size_index]
    policy_rows.append(
        {
            "n_agents": size,
            "Threshold-agent mean": np.nanmean(threshold_payoff),
            "Threshold-agent median": np.nanmedian(threshold_payoff),
            "Treated-BR mean": np.nanmean(treated_peer_payoff),
            "Baseline-BR mean": np.nanmean(baseline_peer_payoff),
            "Threshold minus treated BR": np.nanmean(
                threshold_payoff - treated_peer_payoff
            ),
            "Treated BR minus baseline BR": np.nanmean(
                treated_peer_payoff - baseline_peer_payoff
            ),
            "P(threshold agent > treated BR)": np.nanmean(
                threshold_payoff > treated_peer_payoff
            ),
        }
    )

policy_summary = pd.DataFrame(policy_rows).set_index("n_agents")
policy_summary.style.format(
    {
        "Threshold-agent mean": "{:.4f}",
        "Threshold-agent median": "{:.4f}",
        "Treated-BR mean": "{:.4f}",
        "Baseline-BR mean": "{:.4f}",
        "Threshold minus treated BR": "{:.4f}",
        "Treated BR minus baseline BR": "{:.4f}",
        "P(threshold agent > treated BR)": "{:.1%}",
    }
)

In [ ]:
threshold_bin = pd.cut(
    theta.link_threshold, threshold_edges, labels=False, include_lowest=True
)
fig, axes = plt.subplots(3, 2, figsize=(12, 11), constrained_layout=True)

for size_index, size in enumerate(n_agents):
    rows = []
    for bin_index in range(10):
        selected = np.asarray(threshold_bin == bin_index)
        rows.append(
            {
                "threshold": theta.loc[selected, "link_threshold"].mean(),
                "global_effect": welfare_delta[size_index, selected, -1].mean(),
                "threshold_payoff": np.nanmean(
                    threshold_agent_payoff[size_index, selected]
                ),
                "treated_BR_payoff": np.nanmean(
                    treated_BR_payoff[size_index, selected]
                ),
                "baseline_BR_payoff": np.nanmean(
                    baseline_BR_payoff[size_index, selected]
                ),
            }
        )
    profile = pd.DataFrame(rows)

    left, right = axes[size_index]
    left.plot(profile.threshold, profile.global_effect, color=SIZE_COLORS[size], lw=2)
    left.axhline(0, color="black", lw=0.8)
    left.set_ylabel("paired aggregate-welfare effect")
    left.set_title(f"n = {size}: aggregate effect")

    right.plot(profile.threshold, profile.threshold_payoff, lw=2, label="threshold agent")
    right.plot(profile.threshold, profile.treated_BR_payoff, lw=2, label="treated BR peers")
    right.plot(profile.threshold, profile.baseline_BR_payoff, lw=2, ls="--", label="all-BR baseline")
    right.axhline(0, color="black", lw=0.8)
    right.set_ylabel("policy-specific normalized payoff")
    right.set_title(f"n = {size}: who gains?")

    left.set_xlabel("link threshold")
    right.set_xlabel("link threshold")

axes[0, 1].legend(frameon=False)
fig.suptitle("Aggregate welfare and policy-specific payoff by threshold")
plt.show()

## 8. Conclusions for This Artifact

- The configured treatment is harmful at the common horizon for $n=10$, but beneficial on average for $n=50$ and $n=100$.
- The $n=10$ effect is temporarily positive before reversing; larger-network gains emerge later.
- Threshold effects are nonlinear and interact strongly with visibility and outside optimism; a single OLS slope is not an adequate summary.
- Welfare changes track changes in centralization much more closely than changes in link abundance.
- At the endpoint, the threshold agent underperforms treated best-response peers for $n=10$ but outperforms them for $n=50$ and $n=100$.
- Cross-size differences cannot be attributed to the threshold rule alone: at the common endpoint, the threshold agent's expected activation count rises from 40 to 200 and 400 as $n$ rises, while a baseline agent receives 40 activations.

## Reporting Notes

- Parameter-row ranges and proportions describe the fixed design; they are not sampling confidence intervals.
- Episode outputs were stored as 16-episode means, so episode-level simulation variance is not available.
- Normalized time is population-average activation events per agent. With unequal weights it is not equal individual exposure.
- The treatment jointly changes policy assignment and activation weight, so it does not identify a policy-only replacement effect.
- This is a space-filling Sobol QMC design, not the Saltelli matrix required for conventional Sobol sensitivity indices.
- The nonlinear surrogate is assessed only as a predictive description of the available effect surface.
- The artifact uses $2^{14}=16{,}384$ parameter rows and stores centralization and policy welfare; paper methods describing a $2^{12}$ benchmark run should not be reused for this analysis without revision.